# **Extraction hebdomadaire**

In [ ]:
import os, re, json, shutil, base64, itertools, unicodedata, html as htmllib, requests, urllib3, pandas as pd, openpyxl
from getpass import getpass
from datetime import datetime
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files, drive

drive.mount('/content/drive')

# ============ 1. CONFIG ============
DOSSIER = "/content/drive/MyDrive/Extraction_hebdo"
GABARIT = f"{DOSSIER}/Situation_completude_par_region_19_10_2026.xlsx"   # canevas : structure uniquement
aujourdhui = datetime.now()
SORTIE = f"{DOSSIER}/Situation_completude_par_region_{aujourdhui.strftime('%d_%m_%Y')}.xlsx"

# --- mémoire des extractions + dashboard ---
HISTORIQUE   = f"{DOSSIER}/Historique_extractions.csv"            # mémoire de TOUTES les extractions
GEO_CACHE    = f"{DOSSIER}/geometries_districts.geojson"           # contours des districts (téléchargés 1 seule fois)
GEO_LOCAL    = f"{DOSSIER}/districts_senegal.geojson"              # optionnel : votre propre fichier de contours
DASHBOARD    = f"{DOSSIER}/Dashboard_completude.html"             # DASHBOARD UNIQUE : toujours le même fichier
ARCHIVER_DASHBOARD = False   # True = garde aussi une copie datée dans le sous-dossier Archives_dashboard
LOGO         = f"{DOSSIER}/logo_MSHP.jpg"                          # logo du ministère (si introuvable : logo.png / logo_MSHP.png / logo.jpg du même dossier)
ENTETE = ["Ministère de la Santé et de l'Hygiène publique",
          "Direction de la Planification, de la Recherche et des Statistiques",
          "Division du Système d'Information Sanitaire et Sociale"]
TITRE_DASHBOARD = "Tableau de bord de suivi de la complétude des rapports DHIS2"
SEUIL_CRITIQUE, SEUIL_CONFORME = 70, 90   # < 70 critique ; 70-90 à surveiller ; >= 90 conforme
SEUIL_BAISSE = 5                          # écart (en points) à partir duquel une baisse/hausse est signalée
RECHARGER_CONTOURS = False   # True = retélécharger les contours depuis DHIS2 même s'ils sont en cache
NB_EXTRACTIONS_DETAIL = 8    # nb d'extractions gardées au niveau district dans le dashboard
MOIS_GENERES = [7, 8, 9]     # mois créés si le canevas n'a pas de période (ajoutez 10 pour octobre)
IMPORTER_ANCIENNES = True    # reprend dans l'historique les anciens fichiers Situation_completude_par_region_JJ_MM_AAAA.xlsx

# --- NOUVEAU : sous-dossier d'archives + dates à retirer de la mémoire ---
# Les fichiers Excel datés du dossier principal ET de TOUS ses sous-dossiers sont scannés et gardés en mémoire
# (c'est ce qui alimente l'évolution par extraction). Aujourd'hui (extraction fraîche) est toujours conservé.
DOSSIER_ARCHIVES = f"{DOSSIER}/Archives_extractions"   # NOM DE VOTRE NOUVEAU SOUS-DOSSIER (à adapter) : l'Excel du jour y sera aussi copié
DATES_A_EXCLURE = ["2026-11-06", "2026-11-07", "2026-11-08"]   # extractions retirées de la mémoire (format AAAA-MM-JJ)

# Suppression des exclusions de mois
MOIS_EXCLUS = set()
MOIS_EXCLUS_AVANT_15 = set()

USERNAME = input("Identifiant DHIS2 : ")
PASSWORD = getpass("Mot de passe DHIS2 : ")
URLS_POSSIBLES = ["https://senegal.dhis2.org/dhis", "https://senegal.dhis2.org"]

if not os.path.exists(GABARIT):
    raise SystemExit(f"❌ Gabarit introuvable : {GABARIT}")

# ============ 2. OUTILS ============
MOIS = {"janvier":1,"fevrier":2,"mars":3,"avril":4,"mai":5,"juin":6,"juillet":7,
        "aout":8,"septembre":9,"octobre":10,"novembre":11,"decembre":12}
MOIS_FR = {1:"Janvier",2:"Février",3:"Mars",4:"Avril",5:"Mai",6:"Juin",7:"Juillet",
           8:"Août",9:"Septembre",10:"Octobre",11:"Novembre",12:"Décembre"}

def norm(s):
    s = unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()

def parse_periode(v):
    """'Juillet 2026', 'Ici Novembre 2026', datetime -> (année, mois)"""
    if pd.isna(v) or str(v).strip() == "": return None
    if isinstance(v, (datetime, pd.Timestamp)): return v.year, v.month
    t = norm(v).split()
    mois = next((MOIS[x] for x in t if x in MOIS), None)
    an = next((int(x) for x in t if x.isdigit() and len(x) == 4), None)
    if mois and an: return an, mois
    d = pd.to_datetime(v, errors="coerce")
    return (d.year, d.month) if pd.notna(d) else None

def mois_garde(p):
    if p is None: return False
    m = p[1]
    if m in MOIS_EXCLUS: return False
    if m in MOIS_EXCLUS_AVANT_15 and (aujourdhui.month, aujourdhui.day) < (m, 15): return False
    return True

# ============ 3. CONNEXION ============
s = requests.Session(); s.auth = (USERNAME, PASSWORD)
BASE_URL = None
for verify in (True, False):
    for url in URLS_POSSIBLES:
        try:
            r = s.get(f"{url}/api/me.json", params={"fields": "username"}, timeout=30, verify=verify)
            if r.status_code == 200 and "username" in r.text:
                BASE_URL, s.verify = url, verify
                break
            print(f"… {url} (SSL={verify}) -> HTTP {r.status_code}")
        except Exception as e:
            print(f"… {url} (SSL={verify}) -> {type(e).__name__}")
    if BASE_URL: break
if not BASE_URL:
    raise SystemExit("❌ Connexion impossible (identifiants faux ou serveur qui bloque Colab). Voir les lignes ci-dessus.")
if not s.verify: urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
print(f"✅ Connecté à {BASE_URL}")

def api(path, **params):
    r = s.get(f"{BASE_URL}/api/{path}", params=params, timeout=180)
    r.raise_for_status()
    return r.json()

# ============ 4. LECTURE DU CANEVAS ============
xl = pd.ExcelFile(GABARIT)
feuilles = {n: pd.read_excel(GABARIT, sheet_name=n) for n in xl.sheet_names}

def cols_dp(df):
    df.columns = df.columns.astype(str).str.strip()
    cd = next((c for c in df.columns if c.lower().startswith("dis")), None)
    cp = next((c for c in df.columns if c.lower().startswith("per") or "per" in c.lower()), None)
    if not cp:
        cp = next((c for c in df.columns if "p" in c.lower() and "ri" in c.lower()), None)
    return cd, cp

for n, df in feuilles.items():
    df.columns = [c.replace("\u0000", "") for c in df.columns.astype(str)]

tous_districts = set()
for n, df in feuilles.items():
    cd, _ = cols_dp(df)
    if cd: tous_districts |= set(df[cd].ffill().dropna().astype(str))

# ============ 5. UIDs : formulaires + districts ============
datasets = {norm(d["displayName"]): d["id"]
            for d in api("dataSets.json", fields="id,displayName", paging="false")["dataSets"]}

def uid_colonne(col):
    nom = re.sub(r"\s*-\s*reporting rate\s*$", "", col, flags=re.I).strip()
    nom_sans_prefixe = re.sub(r"^R\d+_", "", nom).strip()

    n_nom = norm(nom)
    n_sans = norm(nom_sans_prefixe)

    if n_nom in datasets: return datasets[n_nom] + ".REPORTING_RATE"
    if n_sans in datasets: return datasets[n_sans] + ".REPORTING_RATE"

    for k, v in datasets.items():
        if n_sans and (n_sans in k or k in n_sans):
            return v + ".REPORTING_RATE"
    return None

meilleur, ou_by_name, niveau = -1, {}, None
for lvl in (2, 3, 4):
    ous = api("organisationUnits.json", fields="id,displayName", paging="false", level=lvl)["organisationUnits"]
    d = {norm(o["displayName"]): o["id"] for o in ous}
    score = sum(1 for x in tous_districts if norm(x) in d)
    if score > meilleur: meilleur, ou_by_name, niveau = score, d, lvl
print(f"Niveau des districts : {niveau} ({meilleur}/{len(tous_districts)} reconnus)")

def uid_district(nom):
    n = norm(nom)
    if n in ou_by_name: return ou_by_name[n]
    hits = [k for k in ou_by_name if n and (n in k or k in n)]
    return ou_by_name[hits[0]] if len(hits) == 1 else None

UID_DIST = {}   # (région, district) -> UID DHIS2, pour relier l'historique à la carte

# ============ 6. EXTRACTION DEPUIS DHIS2 ============
resultats, col_ko, dist_ko = {}, set(), set()
DIAG_REGIONS = []   # diagnostic par région pour l'onglet « Qualité » du dashboard

for region, df_orig in feuilles.items():
    cd, cp = cols_dp(df_orig)
    if not cd:
        print(f"☐ {region} : colonne Districts introuvable"); continue

    cols_indicateurs = [c for c in df_orig.columns if c not in (cd, cp) and not c.startswith("Unnamed:")]

    if not cp or cp not in df_orig.columns or df_orig[cp].isna().all():
        annee_courante = aujourdhui.year
        districts_uniques = df_orig[cd].ffill().dropna().unique()
        nouvelles_lignes = []
        for dist in districts_uniques:
            for m in MOIS_GENERES:
                ligne = {"Districts": dist, "Période": f"{MOIS_FR[m]} {annee_courante}"}
                for col_ind in cols_indicateurs:
                    ligne[col_ind] = None
                nouvelles_lignes.append(ligne)
        df = pd.DataFrame(nouvelles_lignes)
        cp = "Période"
        cd = "Districts"
    else:
        df = df_orig.rename(columns={cd: "Districts", cp: "Période"})
        df["Districts"] = df["Districts"].ffill()

    print(f" {region} : périodes du canevas = {list(df['Période'].dropna().astype(str).unique())[:12]}")

    periodes = [parse_periode(v) for v in df["Période"]]
    garder = [mois_garde(p) for p in periodes]
    df = df.loc[garder].copy()
    periodes = [p for p, g in zip(periodes, garder) if g]
    if df.empty:
        print(f"☐ {region} : aucune ligne après filtrage temporel"); continue
    df = df.reset_index(drop=True)
    df["_p"] = pd.Series(periodes, dtype="object")

    cols = [c for c in df.columns if c not in ("Districts", "Période", "_p")]
    dx = {c: uid_colonne(c) for c in cols}
    ou = {d: uid_district(d) for d in df["Districts"].dropna().unique()}
    UID_DIST.update({(region, d): u for d, u in ou.items() if u})
    col_ko |= {c for c, u in dx.items() if not u}
    dist_ko |= {d for d, u in ou.items() if not u}
    pe = sorted({f"{y}{m:02d}" for y, m in periodes})
    dx_ids = sorted({u for u in dx.values() if u}); ou_ids = sorted({u for u in ou.values() if u})

    valeurs = {}
    if dx_ids and ou_ids and pe:
        try:
            a = api("analytics.json",
                    dimension=[f"dx:{';'.join(dx_ids)}", f"ou:{';'.join(ou_ids)}", f"pe:{';'.join(pe)}"],
                    skipMeta="true")
            h = [x["name"] for x in a["headers"]]
            for row in a["rows"]:
                r = dict(zip(h, row)); valeurs[(r["dx"], r["ou"], r["pe"])] = float(r["value"])
        except Exception as e:
            print(f"☐ {region} : erreur Analytics : {e}")

    for c in cols:
        df[c] = pd.Series([valeurs.get((dx[c], ou.get(d), f"{p[0]}{p[1]:02d}")) if dx[c] and ou.get(d) else None
                           for d, p in zip(df["Districts"], periodes)], dtype="float64")
    df["Période"] = [f"{MOIS_FR[p[1]]} {p[0]}" for p in periodes]
    resultats[region] = df.drop(columns="_p")
    print(f" {region} : {len(valeurs)} valeurs récupérées sur {len(df)*len(cols)} cases")
    DIAG_REGIONS.append({"region": region, "valeurs": len(valeurs), "cases": len(df) * len(cols)})

if not resultats:
    raise SystemExit("❌ Rien à écrire : aucune feuille valide (canevas sans périodes exploitables).")

with pd.ExcelWriter(SORTIE, engine="openpyxl") as w:
    for n, d in resultats.items(): d.to_excel(w, sheet_name=n[:31], index=False)

if col_ko:  print("\n☐ Colonnes sans formulaire DHIS2 correspondant :", *sorted(col_ko), sep="\n  - ")
if dist_ko: print("\n☐ Districts non reconnus dans DHIS2 :", *sorted(dist_ko), sep="\n  - ")

# ============ 7. MISE EN FORME ============
F = lambda **k: Font(name="Segoe UI", **k)
fill = lambda c: PatternFill(start_color=c, end_color=c, fill_type="solid")
side = Side(style="thin", color="D9D9D9"); bord = Border(left=side, right=side, top=side, bottom=side)
wb = openpyxl.load_workbook(SORTIE)
for ws in wb.worksheets:
    R, C = ws.max_row, ws.max_column
    ws.row_dimensions[1].height = 28
    for c in range(1, C + 1):
        x = ws.cell(1, c); x.font = F(size=11, bold=True, color="FFFFFF"); x.fill = fill("1F4E78")
        x.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True); x.border = bord
    for r in range(2, R + 1):
        ws.row_dimensions[r].height = 20
        for c in range(1, C + 1):
            x = ws.cell(r, c); x.font = F(size=10); x.border = bord
            x.alignment = Alignment(horizontal="left" if c == 1 else "center", vertical="center", wrap_text=(c == 1))
            if c >= 3 and isinstance(x.value, (int, float)):
                x.number_format = "0.0"
                if x.value < SEUIL_CRITIQUE:   x.fill, x.font = fill("FF0000"), F(size=10, bold=True, color="FFFFFF")
                elif x.value < SEUIL_CONFORME: x.fill, x.font = fill("FFC000"), F(size=10, bold=True, color="000000")
                else:              x.fill, x.font = fill("00B050"), F(size=10, bold=True, color="FFFFFF")
    r = 2
    while r <= R:
        v, e = ws.cell(r, 1).value, r
        while e + 1 <= R and ws.cell(e + 1, 1).value == v: e += 1
        if e > r and v is not None:
            for k in range(r + 1, e + 1): ws.cell(k, 1).value = None
            ws.merge_cells(start_row=r, start_column=1, end_row=e, end_column=1)
        r = e + 1
    for c in range(1, C + 1):
        L = max(len(str(ws.cell(r, c).value or "")) for r in range(1, R + 1))
        ws.column_dimensions[get_column_letter(c)].width = min(max(L + 3, 12), 30)
wb.save(SORTIE)
print(f"\n✨ Excel terminé : {SORTIE}")

# NOUVEAU : copie de l'Excel du jour dans le sous-dossier d'archives (mémoire de l'évolution)
if DOSSIER_ARCHIVES:
    os.makedirs(DOSSIER_ARCHIVES, exist_ok=True)
    shutil.copy(SORTIE, os.path.join(DOSSIER_ARCHIVES, os.path.basename(SORTIE)))
    print(f"📁 Copie de l'Excel du jour dans : {DOSSIER_ARCHIVES}")

# =====================================================================
# ============ 8. MÉMOIRE : HISTORIQUE DE TOUTES LES EXTRACTIONS =======
# =====================================================================
COLS_HIST = ["date_extraction", "region", "district", "uid_district",
             "periode_code", "periode", "formulaire", "valeur"]

def nom_formulaire(col):
    return re.sub(r"\s*-\s*reporting rate\s*$", "", str(col), flags=re.I).strip()

def libelle_periode(code):
    """'202607' -> 'Juillet 2026' : libellé toujours propre, recalculé depuis le code."""
    code = str(code).strip()
    if re.fullmatch(r"\d{6}", code) and 1 <= int(code[4:6]) <= 12:
        return f"{MOIS_FR[int(code[4:6])]} {code[:4]}"
    return ""

def nettoyer_hist(hist):
    """Corrige les NaN (texte vide relu comme float) et écarte les lignes sans période ou sans formulaire."""
    hist = hist.copy()
    for c in ("date_extraction", "region", "district", "uid_district", "periode_code", "periode", "formulaire"):
        hist[c] = hist[c].fillna("").astype(str).str.strip()
    hist["periode"] = hist["periode_code"].map(libelle_periode)   # « Juillet 2026 », « Août 2026 »…
    ok = (hist["periode"] != "") & (hist["formulaire"] != "")
    mauvaises = hist[~ok]
    if len(mauvaises):
        print(f"☐ {len(mauvaises)} ligne(s) sans période/formulaire ignorées (dates : {sorted(mauvaises['date_extraction'].unique())})")
    return hist[ok].reset_index(drop=True)

def construire_lignes(resultats, uid_dist, date_ext):
    """Transforme {région: tableau} en lignes 'longues' (une ligne = district x période x formulaire)."""
    lignes = []
    for region, d in resultats.items():
        cols_f = [c for c in d.columns if c not in ("Districts", "Période")]
        for _, row in d.iterrows():
            p = parse_periode(row["Période"])
            pcode = f"{p[0]}{p[1]:02d}" if p else ""
            for c in cols_f:
                v = row[c]
                lignes.append({"date_extraction": date_ext, "region": region, "district": row["Districts"],
                               "uid_district": uid_dist.get((region, row["Districts"]), ""),
                               "periode_code": pcode, "periode": libelle_periode(pcode),
                               "formulaire": nom_formulaire(c),
                               "valeur": None if pd.isna(v) else round(float(v), 2)})
    return pd.DataFrame(lignes, columns=COLS_HIST)

def ecrire_historique(hist, chemin):
    hist["uid_district"] = hist["uid_district"].fillna("")
    hist["periode_code"] = hist["periode_code"].fillna("")
    hist = hist.sort_values(["date_extraction", "region", "district", "periode_code", "formulaire"]).reset_index(drop=True)
    hist.to_csv(chemin, index=False, encoding="utf-8-sig")
    return hist

def maj_historique(resultats, uid_dist, date_ext, chemin):
    nouveau = construire_lignes(resultats, uid_dist, date_ext)
    remplace = 0
    if os.path.exists(chemin):
        ancien = pd.read_csv(chemin, dtype={"periode_code": str, "uid_district": str, "date_extraction": str},
                             encoding="utf-8-sig")
        ancien["periode_code"] = ancien["periode_code"].fillna("")
        ancien["uid_district"] = ancien["uid_district"].fillna("")
        shutil.copy(chemin, chemin.replace(".csv", "_sauvegarde.csv"))   # filet de sécurité
        remplace = int((ancien["date_extraction"] == date_ext).sum())
        ancien = ancien[ancien["date_extraction"] != date_ext]
        hist = pd.concat([ancien, nouveau], ignore_index=True)
    else:
        hist = nouveau
    return ecrire_historique(hist, chemin), remplace

def purger_dates(hist, dates, chemin, date_courante):
    """Retire de la mémoire les extractions listées (jamais la date du jour)."""
    a_retirer = [d for d in dates if d != date_courante]
    n = int(hist["date_extraction"].isin(a_retirer).sum())
    if n:
        hist = ecrire_historique(hist[~hist["date_extraction"].isin(a_retirer)].copy(), chemin)
        print(f"🧹 {n} ligne(s) retirées de la mémoire (extractions exclues : {', '.join(a_retirer)})")
    return hist

def fichiers_datees(dossier):
    """Liste (chemin, date AAAA-MM-JJ) des Excel Situation_completude_par_region_JJ_MM_AAAA.xlsx
    du dossier ET de tous ses sous-dossiers (Archives_dashboard ignoré)."""
    modele = re.compile(r"^Situation_completude_par_region_(\d{2})_(\d{2})_(\d{4})\.xlsx$", re.I)
    trouves = []
    for racine, sous, noms in os.walk(dossier):
        sous[:] = [x for x in sous if x.lower() != "archives_dashboard"]
        for nom in sorted(noms):
            m = modele.match(nom)
            if m:
                jj, mm, aaaa = m.groups()
                trouves.append((os.path.join(racine, nom), f"{aaaa}-{mm}-{jj}"))
    return trouves

def importer_anciennes(hist, dossier, uid_dist, date_courante, chemin, dates_exclues=()):
    """Reprend les anciens fichiers Excel datés (dossier + sous-dossiers) qui ne sont pas encore dans l'historique.
    Idempotent. Ignore les dates exclues, la date du jour (extraction fraîche prioritaire)
    et les fichiers sans aucune valeur (canevas)."""
    deja, ajouts, ignores = set(hist["date_extraction"]), [], []
    for chemin_f, date_f in fichiers_datees(dossier):
        nom = os.path.relpath(chemin_f, dossier)
        if date_f in dates_exclues or date_f in deja or date_f == date_courante: continue
        try:
            xf, res_f = pd.ExcelFile(chemin_f), {}
            for feuille in xf.sheet_names:
                df = pd.read_excel(xf, sheet_name=feuille)
                df.columns = [str(c).strip() for c in df.columns]
                if "Districts" in df.columns and "Période" in df.columns:
                    df["Districts"] = df["Districts"].ffill()
                    res_f[feuille] = df.dropna(subset=["Districts"])
            lignes = construire_lignes(res_f, uid_dist, date_f) if res_f else None
        except Exception as e:
            print(f"☐ Import ignoré ({nom}) : {type(e).__name__} {e}"); continue
        if lignes is None or lignes["valeur"].notna().sum() == 0:
            ignores.append(nom); continue
        ajouts.append(lignes); deja.add(date_f)
        print(f"🧠 Ancienne extraction reprise : {nom} ({int(lignes['valeur'].notna().sum())} valeurs)")
    if ignores: print("   Fichiers sans valeur ignorés (canevas ou extraction vide) :", *ignores, sep="\n   - ")
    if not ajouts: return hist
    return ecrire_historique(pd.concat([hist] + ajouts, ignore_index=True), chemin)

date_ext = aujourdhui.strftime("%Y-%m-%d")
hist, remplace = maj_historique(resultats, UID_DIST, date_ext, HISTORIQUE)
if remplace:
    print(f"🧠 Historique : les {remplace} lignes déjà enregistrées pour le {aujourdhui.strftime('%d/%m/%Y')} ont été remplacées.")
hist = purger_dates(hist, DATES_A_EXCLURE, HISTORIQUE, date_ext)            # retire 06, 07, 08/11
if IMPORTER_ANCIENNES:
    hist = importer_anciennes(hist, DOSSIER, UID_DIST, date_ext, HISTORIQUE, set(DATES_A_EXCLURE))
hist = nettoyer_hist(hist)                                                    # corrige le bug NaN/str
print(f"🧠 Historique mis à jour : {hist['date_extraction'].nunique()} extraction(s) en mémoire "
      f"({len(hist)} lignes) -> {HISTORIQUE}")
print("   Extractions en mémoire :", ", ".join(sorted(hist["date_extraction"].unique())))

# =====================================================================
# ============ 9. CONTOURS DES DISTRICTS (pour la carte) ===============
# =====================================================================
def _arrondir(c, nd=4):
    return round(c, nd) if isinstance(c, (int, float)) else [_arrondir(x, nd) for x in c]

def _nom_feature(props):
    for k in ("name", "displayName", "NAME", "Name", "NOM", "nom", "district", "District", "ADM2_FR", "ADM2_EN"):
        if props.get(k): return str(props[k])
    return ""

def _brut_depuis_dhis2(session, base_url, niveau):
    feats = []
    try:
        r = session.get(f"{base_url}/api/organisationUnits.geojson",
                        params={"level": niveau, "paging": "false"}, timeout=240)
        if r.status_code == 200:
            for f in r.json().get("features", []):
                p, g = f.get("properties", {}) or {}, f.get("geometry")
                if g and g.get("type"):
                    feats.append({"id": p.get("id") or f.get("id") or "", "name": _nom_feature(p), "geometry": g})
    except Exception as e:
        print(f"… contours : endpoint GeoJSON indisponible ({type(e).__name__})")
    if feats: return feats
    try:
        r = session.get(f"{base_url}/api/organisationUnits.json",
                        params={"fields": "id,displayName,geometry", "level": niveau, "paging": "false"}, timeout=240)
        if r.status_code == 200:
            for o in r.json().get("organisationUnits", []):
                g = o.get("geometry")
                if isinstance(g, dict) and g.get("type"):
                    feats.append({"id": o["id"], "name": o.get("displayName", ""), "geometry": g})
    except Exception as e:
        print(f"… contours : lecture JSON impossible ({type(e).__name__})")
    return feats

def charger_contours(session, base_url, niveau, hist, cache, local, recharger=False):
    dist = hist[["region", "district", "uid_district"]].drop_duplicates(["region", "district"]).reset_index(drop=True)
    brut = []
    for chemin in ([local] if os.path.exists(local) else []) + ([] if recharger else [cache]):
        if os.path.exists(chemin):
            try:
                with open(chemin, encoding="utf-8") as fh: gj = json.load(fh)
                for f in gj.get("features", []):
                    p = f.get("properties", {}) or {}
                    if f.get("geometry"):
                        brut.append({"id": p.get("id") or p.get("uid") or "", "name": _nom_feature(p), "geometry": f["geometry"]})
                if brut:
                    print(f"🗺️ Contours lus depuis {chemin} ({len(brut)} entités)"); break
            except Exception as e:
                print(f"☐ Contours illisibles dans {chemin} : {e}")
    if not brut:
        brut = _brut_depuis_dhis2(session, base_url, niveau)
        if brut:
            for b in brut: b["geometry"]["coordinates"] = _arrondir(b["geometry"]["coordinates"])
            with open(cache, "w", encoding="utf-8") as fh:
                json.dump({"type": "FeatureCollection", "features": [
                    {"type": "Feature", "properties": {"id": b["id"], "name": b["name"]}, "geometry": b["geometry"]}
                    for b in brut]}, fh, ensure_ascii=False)
            print(f"🗺️ Contours téléchargés depuis DHIS2 ({len(brut)} entités) et mis en cache : {cache}")
    par_uid = {u: i for i, u in enumerate(dist["uid_district"]) if u}
    comptes = dist["district"].map(norm).value_counts()
    par_nom = {norm(n): i for i, n in enumerate(dist["district"]) if comptes[norm(n)] == 1}
    features, vus = [], set()
    for b in brut:
        i = par_uid.get(b["id"], par_nom.get(norm(b["name"])))
        if i is None or i in vus: continue
        vus.add(i)
        g = {"type": b["geometry"]["type"], "coordinates": _arrondir(b["geometry"]["coordinates"])}
        features.append({"type": "Feature", "properties": {"d": int(i)}, "geometry": g})
    sans = [dist.loc[i, "district"] for i in range(len(dist)) if i not in vus]
    print(f"🗺️ {len(features)}/{len(dist)} districts reliés à un contour.")
    if sans: print("☐ Districts sans contour (absents de la carte) :", *sorted(sans), sep="\n  - ")
    return dist, features

# =====================================================================
# ============ 10. DASHBOARD UNIQUE (carte, analyses, alertes) ========
# =====================================================================
DRAPEAU_SVG = ('<svg class="flag" viewBox="0 0 30 20" role="img" aria-label="Drapeau du Sénégal" xmlns="http://www.w3.org/2000/svg">'
               '<rect width="10" height="20" fill="#00853F"/><rect x="10" width="10" height="20" fill="#FDEF42"/>'
               '<rect x="20" width="10" height="20" fill="#E31B23"/>'
               '<polygon fill="#00853F" points="15,6.4 15.81,8.89 18.42,8.89 16.31,10.42 17.12,12.91 15,11.375 12.88,12.91 13.69,10.42 11.58,8.89 14.19,8.89"/></svg>')

def trouver_logo(chemin):
    """Chemin indiqué, sinon recherche d'un fichier logo dans le dossier de travail."""
    if chemin and os.path.exists(chemin): return chemin
    for nom in ("logo_MSHP.jpg", "logo_MSHP.jpeg", "logo_MSHP.png", "logo.png", "logo.jpg", "logo.jpeg", "logo.svg", "logo.webp"):
        c = os.path.join(DOSSIER, nom)
        if os.path.exists(c): return c
    return None

def logo_html(chemin):
    """Logo encodé dans la page (le dashboard reste un seul fichier autonome), allégé si Pillow est disponible."""
    p = trouver_logo(chemin)
    if not p:
        print(f"☐ Logo introuvable ({chemin}) : déposez logo_MSHP.jpg (ou logo.png) dans {DOSSIER}")
        return ""
    ext = os.path.splitext(p)[1].lower().lstrip(".")
    mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg",
            "svg": "image/svg+xml", "webp": "image/webp"}.get(ext)
    if not mime:
        print(f"☐ Format de logo non pris en charge : {p}"); return ""
    try:
        with open(p, "rb") as fh: brut = fh.read()
        if ext in ("png", "jpg", "jpeg"):
            try:
                from PIL import Image
                import io
                im = Image.open(io.BytesIO(brut)); im.thumbnail((480, 480)); buf = io.BytesIO()
                if ext == "png": im.save(buf, "PNG", optimize=True)
                else: im.convert("RGB").save(buf, "JPEG", quality=90)
                brut = buf.getvalue()
            except Exception:
                pass
        print(f"🖼️ Logo intégré à l'entête : {p}")
        return f'<img class="logo" alt="Logo du Ministère" src="data:{mime};base64,{base64.b64encode(brut).decode()}">'
    except Exception as e:
        print(f"☐ Logo illisible ({p}) : {e}"); return ""

def entete_html(lignes, chemin_logo):
    corps = "".join(f'<div class="org org{k + 1}">{htmllib.escape(l)}</div>' for k, l in enumerate(lignes))
    republique = (f'<div class="republique">{DRAPEAU_SVG}<div class="rep-t"><b>République du Sénégal</b>'
                  f'<small>Un Peuple, Un But, Une Foi</small></div></div>')
    return f'{republique}{logo_html(chemin_logo)}<div class="orgs">{corps}</div>'

def preparer_donnees(hist, dist, features, nb_detail, options):
    hist = nettoyer_hist(hist)   # CORRECTIF : plus de NaN dans periode_code (erreur str < float)
    exts = sorted(hist["date_extraction"].unique())
    regions = sorted(hist["region"].unique())
    pmap = hist.drop_duplicates("periode_code").set_index("periode_code")["periode"].to_dict()
    pcodes = sorted(c for c in pmap if c)
    formulaires = sorted(hist["formulaire"].unique())
    ei = {e: i for i, e in enumerate(exts)}; ri = {r: i for i, r in enumerate(regions)}
    pi = {p: i for i, p in enumerate(pcodes)}; fi = {f: i for i, f in enumerate(formulaires)}
    di = {(r, d): i for i, (r, d) in enumerate(zip(dist["region"], dist["district"]))}
    qualite = [{"e": ei[e], "cases": int(len(g)), "ren": int(g["valeur"].notna().sum()),
                "sup100": int((g["valeur"] > 100).sum())} for e, g in hist.groupby("date_extraction")]
    hv = hist.dropna(subset=["valeur"]).copy()
    hv["e"] = hv["date_extraction"].map(ei); hv["r"] = hv["region"].map(ri)
    hv["p"] = hv["periode_code"].map(pi);    hv["f"] = hv["formulaire"].map(fi)
    hv["d"] = [di.get(k) for k in zip(hv["region"], hv["district"])]
    hv = hv.dropna(subset=["e", "r", "p", "f", "d"])
    series = []
    for keep in itertools.product([True, False], repeat=3):
        cols = ["e"] + [c for c, k in zip(["r", "p", "f"], keep) if k]
        g = hv.groupby(cols)["valeur"].agg(["mean", "count"]).reset_index()
        for c, k in zip(["r", "p", "f"], keep):
            if not k: g[c] = -1
        for x in g[["e", "r", "p", "f", "mean", "count"]].itertuples(index=False):
            series.append([int(x.e), int(x.r), int(x.p), int(x.f), round(float(x.mean), 1), int(x.count)])
    garde = set(range(max(0, len(exts) - nb_detail), len(exts)))
    det = hv[hv["e"].isin(garde)]
    detail = [[int(a), int(b), int(c), int(d_), round(float(v), 1)]
              for a, b, c, d_, v in zip(det["e"], det["d"], det["p"], det["f"], det["valeur"])]
    crit, conf, baisse = options["seuils"]
    return {"extractions": exts, "regions": regions,
            "periodes": [{"code": c, "label": pmap[c]} for c in pcodes], "formulaires": formulaires,
            "districts": [{"name": d, "r": int(ri[r]), "uid": u}
                          for r, d, u in zip(dist["region"], dist["district"], dist["uid_district"])],
            "detail": detail, "series": series, "qualite": qualite,
            "seuils": {"critique": crit, "conforme": conf, "baisse": baisse},
            "diag": options.get("diag"),
            "geojson": {"type": "FeatureCollection", "features": features}}

HTML_MODELE = r'''<!DOCTYPE html>
<html lang="fr" data-theme="light"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITRE__</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<style>
:root{--bg:#f3f6fb;--card:#fff;--text:#18242f;--muted:#5c6b7a;--border:#e2e8f0;--primary:#1F4E78;--chip:#eef3f9;--hover:#f1f6fc;
--shadow:0 1px 2px rgba(15,35,60,.06),0 6px 18px rgba(15,35,60,.07);--crit:#c62828;--warn:#9a6a00;--ok:#067a46;--nd:#5c6b7a}
:root[data-theme=dark]{--bg:#0d1520;--card:#15202d;--text:#e7eef6;--muted:#9db0c4;--border:#233447;--primary:#6fb1ea;--chip:#1d2d3f;--hover:#1a2a3b;
--shadow:0 1px 2px rgba(0,0,0,.45),0 8px 22px rgba(0,0,0,.35);--crit:#ff8a80;--warn:#ffd166;--ok:#5be3a0;--nd:#9db0c4}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;font-family:system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif;background:var(--bg);color:var(--text);line-height:1.4}
button{font-family:inherit}
.top{background:linear-gradient(120deg,#0a3558 0%,#1F4E78 55%,#2d6fa5 100%);color:#fff;padding:14px 26px;display:flex;align-items:center;justify-content:space-between;gap:16px;flex-wrap:wrap}
.tricolore{height:5px;background:linear-gradient(90deg,#00853F 0 33.33%,#FDEF42 33.33% 66.66%,#E31B23 66.66% 100%)}
.brand{display:flex;align-items:center;gap:20px;flex-wrap:wrap}
.republique{display:flex;align-items:center;gap:12px;padding-right:20px;border-right:1px solid rgba(255,255,255,.3)}
.flag{height:42px;width:auto;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.35)}
.rep-t{line-height:1.3}.rep-t b{display:block;font-size:13px;letter-spacing:.5px;text-transform:uppercase}
.rep-t small{font-size:11px;opacity:.85;font-style:italic}
.orgs{display:flex;flex-direction:column;gap:2px;max-width:640px}
.logo{height:66px;width:auto;max-width:130px;object-fit:contain;background:#fff;border-radius:12px;padding:5px;box-shadow:0 2px 10px rgba(0,0,0,.25)}
.org{line-height:1.3}.org1{font-size:19px;font-weight:700;letter-spacing:.2px}.org2{font-size:13px;opacity:.95}.org3{font-size:12.5px;opacity:.85}
.top-right{display:flex;align-items:center;gap:10px;flex-wrap:wrap}
.badge{background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.3);padding:6px 12px;border-radius:999px;font-size:12px}
.ghost{background:rgba(255,255,255,.14);color:#fff;border:1px solid rgba(255,255,255,.35);padding:7px 12px;border-radius:9px;font-size:12px;cursor:pointer}
.ghost:hover{background:rgba(255,255,255,.26)}
.titlebar{padding:18px 26px 4px}
.titlebar h1{margin:0;font-size:24px;letter-spacing:-.2px}
.titlebar p{margin:4px 0 0;color:var(--muted);font-size:13px}
.sticky{position:sticky;top:0;z-index:1200;background:var(--bg);background:color-mix(in srgb,var(--bg) 88%,transparent);backdrop-filter:blur(8px);border-bottom:1px solid var(--border)}
.filters{display:flex;flex-wrap:wrap;gap:12px;padding:12px 26px 8px;align-items:flex-end}
.filters label{font-size:11px;font-weight:600;color:var(--muted);display:flex;flex-direction:column;gap:4px;text-transform:uppercase;letter-spacing:.4px}
select,input{padding:8px 10px;border:1px solid var(--border);border-radius:9px;font-size:13px;min-width:160px;background:var(--card);color:var(--text)}
select:focus,input:focus,button:focus-visible{outline:2px solid var(--primary);outline-offset:1px}
.btn{padding:8px 13px;border:0;border-radius:9px;background:var(--primary);color:#fff;font-size:12px;cursor:pointer;font-weight:600}
:root[data-theme=dark] .btn{color:#0d1520}
.btn.sec{background:var(--chip);color:var(--text);border:1px solid var(--border)}
.tabs{display:flex;gap:4px;padding:0 26px;overflow-x:auto}
.tab{background:none;border:0;padding:10px 14px;font-size:13px;font-weight:600;color:var(--muted);cursor:pointer;border-bottom:3px solid transparent;white-space:nowrap}
.tab.on{color:var(--primary);border-bottom-color:var(--primary)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(165px,1fr));gap:12px;padding:16px 26px 4px}
.kpi{background:var(--card);border-radius:14px;padding:13px 15px;box-shadow:var(--shadow);border:1px solid var(--border);border-top:4px solid var(--primary)}
.kpi .l{font-size:11px;color:var(--muted);text-transform:uppercase;letter-spacing:.5px;font-weight:600}
.kpi .v{font-size:28px;font-weight:750;margin-top:2px;letter-spacing:-.5px}
.kpi .d{font-size:12px;margin-top:2px;color:var(--muted)}
main{padding:12px 26px 26px}
.panel{display:none}.panel.on{display:block}
.grid2{display:grid;grid-template-columns:1.55fr 1fr;gap:14px}
.grid2e{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:14px}
.stack{display:flex;flex-direction:column;gap:14px}
.card{background:var(--card);border-radius:14px;padding:15px;box-shadow:var(--shadow);border:1px solid var(--border);margin-bottom:0}
.panel>.card{margin-bottom:14px}
.card h2{margin:0 0 10px;font-size:14.5px;color:var(--primary);letter-spacing:.1px}
.card-h{display:flex;align-items:baseline;justify-content:space-between;gap:10px;flex-wrap:wrap}
.hint,.muted{font-size:12px;color:var(--muted)}
#map{height:600px;border-radius:10px;background:var(--chip);z-index:1}
.chartbox{position:relative;height:250px}.chartbox.tall{height:380px}.chartbox.sm{height:190px}
.note{font-size:12px;color:var(--muted);padding:6px 2px}
.tbar{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:10px}
.tbar h2{margin:0;flex:1;min-width:160px}
.tw{overflow:auto;max-height:520px}.tw.free{max-height:none}
table{border-collapse:collapse;width:100%;font-size:13px}
th{position:sticky;top:0;background:var(--primary);color:#fff;text-align:left;padding:9px 10px;cursor:pointer;white-space:nowrap;font-size:12px;z-index:2}
:root[data-theme=dark] th{color:#0d1520}
td{padding:7px 10px;border-bottom:1px solid var(--border)}
tbody tr:hover{background:var(--hover)}
tr.clic{cursor:pointer}
td.c{text-align:center;font-weight:700;border-radius:0}
.pill{display:inline-flex;align-items:center;gap:6px;padding:2px 9px;border-radius:999px;font-size:11.5px;font-weight:650;background:var(--chip)}
.pill::before{content:"";width:8px;height:8px;border-radius:50%;background:currentColor}
.pill.crit{color:var(--crit)}.pill.warn{color:var(--warn)}.pill.ok{color:var(--ok)}.pill.nd{color:var(--nd)}
.legende{display:flex;gap:14px;flex-wrap:wrap;font-size:12px;color:var(--muted)}
.legende i{display:inline-block;width:13px;height:13px;border-radius:3px;vertical-align:-2px;margin-right:5px}
.al-g{margin-bottom:12px}
.al-t{display:flex;justify-content:space-between;align-items:center;font-size:12.5px;font-weight:700;margin-bottom:6px;gap:8px}
.al-t span.n{background:var(--chip);border-radius:999px;padding:1px 9px;font-size:11.5px}
.al{display:flex;width:100%;align-items:center;justify-content:space-between;gap:8px;text-align:left;background:var(--chip);border:1px solid transparent;border-radius:9px;padding:6px 10px;margin-bottom:4px;font-size:12.5px;color:var(--text);cursor:pointer}
.al:hover{border-color:var(--primary)}
.al small{color:var(--muted)}
.al .r{font-weight:700;white-space:nowrap}
.links{font-size:12px;margin-top:2px}.links button{background:none;border:0;color:var(--primary);cursor:pointer;font-weight:600;padding:2px 6px;font-size:12px}
.heat{font-size:12px}.heat th{position:static;cursor:default}
.heat th.rot{height:150px;vertical-align:bottom;padding:6px 4px;font-weight:600}
.heat th.rot div{writing-mode:vertical-rl;transform:rotate(180deg);white-space:nowrap;max-height:140px;overflow:hidden;text-overflow:ellipsis}
.heat td.c{padding:6px 4px;min-width:42px}
.heat td.rl{font-weight:650;white-space:nowrap}
.prose{font-size:13px;color:var(--text)}.prose ul{margin:6px 0 0 18px;padding:0}.prose li{margin:3px 0}
#scrim{position:fixed;inset:0;background:rgba(5,12,20,.45);display:none;z-index:1900}
#scrim.on{display:block}
#drawer{position:fixed;top:0;right:0;height:100%;width:min(460px,100%);background:var(--card);box-shadow:-8px 0 30px rgba(0,0,0,.25);z-index:2000;transform:translateX(105%);transition:transform .22s ease;overflow:auto;padding:18px}
#drawer.on{transform:none}
.dh{display:flex;justify-content:space-between;gap:10px;margin-bottom:10px}
.dh h3{margin:0;font-size:20px}
#dClose{background:var(--chip);border:1px solid var(--border);border-radius:9px;width:34px;height:34px;cursor:pointer;color:var(--text);font-size:15px}
.dk{display:flex;gap:14px;align-items:flex-end;margin:6px 0 12px;flex-wrap:wrap}
.dk .big{font-size:40px;font-weight:800;letter-spacing:-1px;line-height:1}
.toast{position:fixed;bottom:22px;left:50%;transform:translateX(-50%);background:#16212e;color:#fff;padding:9px 16px;border-radius:10px;font-size:13px;z-index:3000;opacity:0;pointer-events:none;transition:opacity .2s}
.toast.on{opacity:1}
.map-btn{background:var(--card);color:var(--text);border:2px solid rgba(0,0,0,.25);border-radius:5px;width:30px;height:30px;font-size:16px;cursor:pointer;line-height:1}
.leg{background:var(--card);color:var(--text);padding:8px 10px;border-radius:9px;box-shadow:var(--shadow);font-size:12px;line-height:1.7}
.leg i{display:inline-block;width:13px;height:13px;border-radius:3px;margin-right:6px;vertical-align:-2px}
.leaflet-container{font-family:inherit}
#banner{padding:0 26px}
.banner{margin-top:10px;padding:9px 14px;border-radius:10px;font-size:12.5px;border:1px solid var(--border);background:var(--chip)}
.banner.warn{border-color:#FFC000;background:rgba(255,192,0,.14)}
.banner.err{border-color:var(--crit);background:rgba(198,40,40,.1)}
.prose ul.synth{margin:0;padding:0;display:grid;gap:8px;grid-template-columns:repeat(auto-fit,minmax(320px,1fr))}
.synth li{list-style:none;background:var(--chip);border-radius:10px;padding:9px 12px;font-size:13px}
.legende{margin-top:8px}
#legMap{margin-top:10px;padding-top:9px;border-top:1px solid var(--border)}
@media print{.banner{display:none}}
footer{padding:0 26px 24px;font-size:11.5px;color:var(--muted)}
@media(max-width:1050px){.grid2,.grid2e{grid-template-columns:1fr}#map{height:460px}}
@media(max-width:720px){.republique{border-right:0;padding-right:0}.org1{font-size:16px}.brand{gap:12px}}
@media print{.sticky,.top-right,#drawer,#scrim,.links,.tbar button,.tbar input,.tbar select{display:none!important}
.panel{display:block!important}body{background:#fff}.card,.kpi{box-shadow:none}.tw{max-height:none}#map{height:420px}}
</style></head><body>
<div class="tricolore" aria-hidden="true"></div>
<header class="top">
 <div class="brand">__ENTETE__</div>
 <div class="top-right"><span class="badge" id="badgeDate"></span>
  <button class="ghost" id="btnTheme" type="button">☾ Mode sombre</button>
  <button class="ghost" id="btnPrint" type="button">⎙ Imprimer / PDF</button></div>
</header>
<div class="titlebar"><h1>__TITRE__</h1><p id="sub"></p></div>
<div id="banner" role="status"></div>
<div class="sticky">
 <div class="filters" role="group" aria-label="Filtres">
  <label>Extraction<select id="selE"></select></label>
  <label>Comparer à<select id="selC"></select></label>
  <label>Région<select id="selR"></select></label>
  <label>Période<select id="selP"></select></label>
  <label>Formulaire<select id="selF"></select></label>
  <button class="btn sec" id="btnReset" type="button">Réinitialiser</button>
  <button class="btn sec" id="btnLink" type="button">🔗 Copier le lien</button>
 </div>
 <nav class="tabs" role="tablist" aria-label="Sections">
  <button class="tab on" role="tab" data-tab="vue" aria-selected="true" type="button">Vue d'ensemble</button>
  <button class="tab" role="tab" data-tab="reg" aria-selected="false" type="button">Régions et formulaires</button>
  <button class="tab" role="tab" data-tab="evo" aria-selected="false" type="button">Évolution</button>
  <button class="tab" role="tab" data-tab="dis" aria-selected="false" type="button">Districts</button>
  <button class="tab" role="tab" data-tab="qua" aria-selected="false" type="button">Qualité et méthode</button>
 </nav>
</div>
<section class="kpis" id="kpis" aria-live="polite"></section>
<main>
 <section class="panel on" id="p-vue">
  <div class="card"><div class="card-h"><h2>Synthèse automatique</h2><span class="hint" id="synCtx"></span></div><div id="synth" class="prose"></div></div>
  <div class="grid2">
   <div class="card"><div class="card-h"><h2>Carte des districts</h2><span class="hint">Cliquez sur un district pour ouvrir sa fiche</span></div>
    <div id="map"></div><div class="legende" id="legMap"></div><div class="note" id="mapnote"></div></div>
   <div class="stack">
    <div class="card"><h2>Évolution du taux moyen</h2><div class="chartbox"><canvas id="chEvo"></canvas></div></div>
    <div class="card"><h2>Répartition des districts par statut</h2><div class="chartbox sm"><canvas id="chStat"></canvas></div></div>
    <div class="card"><div class="card-h"><h2>Priorités d'action</h2><span class="hint" id="alHint"></span></div><div id="alertes"></div></div>
   </div>
  </div>
 </section>
 <section class="panel" id="p-reg">
  <div class="grid2e">
   <div class="card"><h2>Taux moyen par région <span class="hint">(cliquer une barre pour filtrer)</span></h2><div class="chartbox tall"><canvas id="chReg"></canvas></div></div>
   <div class="card"><h2>Taux moyen par formulaire <span class="hint">(cliquer une barre pour filtrer)</span></h2><div class="chartbox tall"><canvas id="chForm"></canvas></div></div>
  </div>
  <div class="card"><h2>Carte thermique régions × formulaires <span class="hint">(extraction et période choisies)</span></h2><div class="tw free" id="heat"></div><div class="legende" id="legHeat"></div></div>
  <div class="grid2e">
   <div class="card"><h2>Statuts des districts par région <span class="hint">(tous les districts)</span></h2><div class="chartbox tall"><canvas id="chStatReg"></canvas></div></div>
   <div class="card"><h2>Synthèse par formulaire <span class="hint">(cliquer une ligne pour filtrer)</span></h2><div class="tw" id="tabForm"></div></div>
  </div>
  <div class="grid2e">
   <div class="card"><h2>10 meilleurs districts</h2><div id="top10"></div></div>
   <div class="card"><h2>10 districts les plus faibles</h2><div id="flop10"></div></div>
  </div>
  <div class="grid2e">
   <div class="card"><h2>Synthèse par région</h2><div class="tw free" id="tabReg"></div></div>
   <div class="card"><h2>Taux moyen par période</h2><div class="chartbox"><canvas id="chPer"></canvas></div></div>
  </div>
 </section>
 <section class="panel" id="p-evo">
  <div class="card"><h2>Évolution par région <span class="hint">(cliquer la légende pour masquer une région)</span></h2><div class="chartbox tall"><canvas id="chEvoReg"></canvas></div></div>
  <div class="card"><h2>Historique des extractions</h2><div class="tw free" id="tabEvo"></div></div>
 </section>
 <section class="panel" id="p-dis">
  <div class="card"><div class="tbar"><h2>Détail par district</h2>
   <select id="selS" aria-label="Statut"><option value="all">Tous les statuts</option><option value="crit">Critique</option><option value="warn">À surveiller</option><option value="ok">Conforme</option><option value="nd">Pas de donnée</option></select>
   <input id="q" placeholder="Rechercher un district ou une région…"><button class="btn" id="csv" type="button">Exporter en CSV</button></div>
   <div class="tw"><table><thead><tr><th data-k="region">Région</th><th data-k="name">District</th><th data-k="v">Taux</th><th data-k="delta">Écart</th><th>Tendance</th><th data-k="cl">Statut</th></tr></thead><tbody id="tb"></tbody></table></div></div>
 </section>
 <section class="panel" id="p-qua">
  <div class="grid2e">
   <div class="card"><h2>Cases renseignées par extraction (%)</h2><div class="chartbox"><canvas id="chQual"></canvas></div></div>
   <div class="card"><h2>Diagnostic de la dernière extraction</h2><div id="diag"></div></div>
  </div>
  <div class="card"><h2>Méthode et définitions</h2><div class="prose" id="methode"></div></div>
 </section>
</main>
<aside id="drawer" aria-hidden="true" role="dialog" aria-label="Fiche détaillée du district">
 <div class="dh"><div><h3 id="dTitle"></h3><div class="muted" id="dSub"></div></div><button id="dClose" type="button" aria-label="Fermer">✕</button></div>
 <div class="dk" id="dKpi"></div>
 <h2 style="font-size:13px;color:var(--primary);margin:6px 0">Évolution entre extractions</h2>
 <div class="chartbox sm"><canvas id="chDist"></canvas></div>
 <h2 style="font-size:13px;color:var(--primary);margin:14px 0 6px">Formulaires × périodes (extraction choisie)</h2>
 <div class="tw free" id="dHeat"></div>
</aside>
<div id="scrim"></div><div class="toast" id="toast"></div>
<footer>Taux = taux de rapportage DHIS2 (reporting rate), moyenne simple des cases sélectionnées. Généré le __GENERE__. Données issues de DHIS2, extraites par la Division du Système d'Information Sanitaire et Sociale.</footer>
<script>
(function(){
"use strict";
var DATA=__DATA__;
var E=DATA.extractions,R=DATA.regions,P=DATA.periodes,F=DATA.formulaires,D=DATA.districts,S=DATA.seuils,Q=DATA.qualite,DG=DATA.diag||null;
var COL={rouge:"#FF0000",orange:"#FFC000",vert:"#00B050",gris:"#BFBFBF"};
var PAL=["#1F4E78","#E4572E","#17A672","#8E44AD","#F5A524","#2E86AB","#C2185B","#7CB342","#5D4037","#00838F","#AD1457","#6D4C41","#455A64","#F57C00","#3949AB"];
var LIB={crit:"Critique",warn:"À surveiller",ok:"Conforme",nd:"Pas de donnée"};
function $(id){return document.getElementById(id);}
function classe(v){if(v===null||v===undefined||isNaN(v))return "nd";return v<S.critique?"crit":(v<S.conforme?"warn":"ok");}
function couleur(v){var c=classe(v);return c==="crit"?COL.rouge:(c==="warn"?COL.orange:(c==="ok"?COL.vert:COL.gris));}
function txtOn(v){var c=classe(v);return(c==="warn"||c==="nd")?"#111":"#fff";}
function fmt(v){return(v===null||v===undefined||isNaN(v))?"–":v.toFixed(1).replace(".",",")+" %";}
function fmtDelta(v){if(v===null||v===undefined||isNaN(v))return "–";return(v>0?"+":"")+v.toFixed(1).replace(".",",")+" pts";}
function fmtDate(s){var a=s.split("-");return a[2]+"/"+a[1]+"/"+a[0];}
function esc(s){return String(s).replace(/[&<>"]/g,function(c){return {"&":"&amp;","<":"&lt;",">":"&gt;","\"":"&quot;"}[c];});}
function cut(s,n){s=String(s);return s.length>n?s.slice(0,n-1)+"…":s;}
function ymax(arr){var v=arr.filter(function(x){return x!==null&&x!==undefined;});return v.length?Math.max(100,Math.ceil(Math.max.apply(null,v)/10)*10):100;}
function moyenne(arr){var v=arr.filter(function(x){return x!==null;});return v.length?v.reduce(function(a,b){return a+b;},0)/v.length:null;}

/* ---------- index des données ---------- */
var det={};
DATA.detail.forEach(function(x){var e=x[0],d=x[1];if(!det[e])det[e]={};if(!det[e][d])det[e][d]=[];det[e][d].push([x[2],x[3],x[4]]);});
var detExts=Object.keys(det).map(Number).sort(function(a,b){return a-b;});
var ser={};DATA.series.forEach(function(x){ser[x[0]+"|"+x[1]+"|"+x[2]+"|"+x[3]]=[x[4],x[5]];});
var qmap={};Q.forEach(function(x){qmap[x.e]=x;});
function serv(e,r,p,f){var s=ser[e+"|"+r+"|"+p+"|"+f];return s?s[0]:null;}
function valDistrict(e,d,p,f){var rows=(det[e]&&det[e][d])||[],s=0,n=0;rows.forEach(function(x){if((p<0||x[0]===p)&&(f<0||x[1]===f)){s+=x[2];n++;}});return n?s/n:null;}

/* ---------- état ---------- */
var selE=$("selE"),selC=$("selC"),selR=$("selR"),selP=$("selP"),selF=$("selF"),selS=$("selS");
var TABS=["vue","reg","evo","dis","qua"];
var st={tab:"vue",tri:{k:"v",dir:1},filtre:"",open:null};
var lignes=[],cur=null,derniereVue=[],CH={};
function opt(sel,value,label){var o=document.createElement("option");o.value=value;o.textContent=label;sel.appendChild(o);}
function remplirCmp(){var keep=selC.value;selC.innerHTML="";opt(selC,"auto","Extraction précédente");var e=+selE.value;for(var i=e-1;i>=0;i--)opt(selC,i,fmtDate(E[i])+(det[i]?"":" (KPI seulement)"));
 if(keep&&[].some.call(selC.options,function(o){return o.value===keep;}))selC.value=keep;}
detExts.slice().reverse().forEach(function(e){opt(selE,e,fmtDate(E[e])+(e===E.length-1?" (dernière)":""));});
opt(selR,-1,"Toutes les régions");R.forEach(function(x,i){opt(selR,i,x);});
opt(selP,-1,"Toutes les périodes");P.forEach(function(x,i){opt(selP,i,x.label);});
opt(selF,-1,"Tous les formulaires (moyenne)");F.forEach(function(x,i){opt(selF,i,x);});
remplirCmp();
$("badgeDate").textContent="Dernière extraction : "+fmtDate(E[E.length-1]);
$("sub").textContent=E.length+" extraction(s) en mémoire ("+fmtDate(E[0])+" → "+fmtDate(E[E.length-1])+") · "+D.length+" districts · "+R.length+" régions · "+F.length+" formulaire(s)";

function calculer(){
 var e=+selE.value,r=+selR.value,p=+selP.value,f=+selF.value,ci=null;
 if(selC.value==="auto"){ci=e-1>=0?e-1:null;}else{ci=+selC.value;}
 lignes=D.map(function(d,i){var v=valDistrict(e,i,p,f),pv=(ci===null)?null:valDistrict(ci,i,p,f);
  return{i:i,region:R[d.r],name:d.name,v:v,delta:(v!==null&&pv!==null)?v-pv:null,cl:classe(v),inReg:(r<0||d.r===r)};});
 cur={e:e,r:r,p:p,f:f,ci:ci};return cur;
}

/* ---------- thème ---------- */
function isDark(){return document.documentElement.getAttribute("data-theme")==="dark";}
function th(){var d=isDark();return{txt:d?"#cbd5e1":"#475569",grid:d?"rgba(148,163,184,.18)":"rgba(100,116,139,.16)"};}
function mk(id,type,data,mut){
 if(typeof Chart==="undefined")return null;
 if(CH[id]){try{CH[id].destroy();}catch(e){}CH[id]=null;}
 var el=$(id);if(!el)return null;var t=th();
 var o={responsive:true,maintainAspectRatio:false,animation:{duration:250},
  plugins:{legend:{labels:{color:t.txt}}},
  scales:{x:{ticks:{color:t.txt},grid:{color:t.grid}},y:{ticks:{color:t.txt},grid:{color:t.grid}}}};
 if(mut)mut(o);
 CH[id]=new Chart(el,{type:type,data:data,options:o});return CH[id];
}
function lineSets(labels,vals){
 return[{label:"Taux moyen",data:vals,borderColor:"#1F4E78",backgroundColor:"#1F4E78",tension:.25,spanGaps:true,pointRadius:4,borderWidth:2.5},
  {label:"Seuil conforme ("+S.conforme+" %)",data:labels.map(function(){return S.conforme;}),borderColor:COL.vert,borderDash:[6,4],pointRadius:0,borderWidth:1.5},
  {label:"Seuil critique ("+S.critique+" %)",data:labels.map(function(){return S.critique;}),borderColor:COL.rouge,borderDash:[6,4],pointRadius:0,borderWidth:1.5}];
}

/* ---------- messages ---------- */
var toastT=null;
function toast(m){var t=$("toast");t.textContent=m;t.classList.add("on");clearTimeout(toastT);toastT=setTimeout(function(){t.classList.remove("on");},2200);}
function copier(txt,msg){msg=msg||"Liste copiée";
 function repli(){try{var a=document.createElement("textarea");a.value=txt;document.body.appendChild(a);a.select();var ok=document.execCommand&&document.execCommand("copy");document.body.removeChild(a);toast(ok?msg:"Copie impossible");}catch(e){toast("Copie impossible");}}
 if(navigator.clipboard&&navigator.clipboard.writeText){navigator.clipboard.writeText(txt).then(function(){toast(msg);},repli);}else{repli();}
}
function telecharger(nom,lignesCsv){var blob=new Blob(["\ufeff"+lignesCsv.join("\n")],{type:"text/csv;charset=utf-8"});var a=document.createElement("a");a.href=URL.createObjectURL(blob);a.download=nom;a.click();}
function csvNum(v){return v===null||v===undefined?"":v.toFixed(1).replace(".",",");}
function contexte(c){return"extraction du "+fmtDate(E[c.e])+" · "+(c.r<0?"toutes les régions":R[c.r])+" · "+(c.p<0?"toutes les périodes":P[c.p].label)+" · "+(c.f<0?"tous les formulaires":F[c.f]);}

/* ---------- KPI ---------- */
function rendreKpis(c){
 var L2=lignes.filter(function(x){return x.inReg;});
 var m=serv(c.e,c.r,c.p,c.f),mp=c.ci!==null?serv(c.ci,c.r,c.p,c.f):null;
 var nR=L2.filter(function(x){return x.cl==="crit";}).length,nO=L2.filter(function(x){return x.cl==="warn";}).length,
     nV=L2.filter(function(x){return x.cl==="ok";}).length,nN=L2.filter(function(x){return x.cl==="nd";}).length;
 var dm=(m!==null&&mp!==null)?m-mp:null;
 var dtxt=dm===null?"Pas d'extraction précédente comparable":(dm>=0?"▲ ":"▼ ")+fmtDelta(dm)+" vs "+fmtDate(E[c.ci]);
 var q=qmap[c.e],qp=q&&q.cases?(100*q.ren/q.cases):null;
 var h="";
 function k(l,v,d,col){h+="<div class='kpi' style='border-top-color:"+col+"'><div class='l'>"+l+"</div><div class='v'>"+v+"</div><div class='d'>"+d+"</div></div>";}
 k("Taux moyen",fmt(m),dtxt,couleur(m));
 k("Districts critiques",nR,"taux &lt; "+S.critique+" %",COL.rouge);
 k("À surveiller",nO,"taux "+S.critique+"–"+S.conforme+" %",COL.orange);
 k("Conformes",nV,"taux ≥ "+S.conforme+" %",COL.vert);
 k("Sans donnée",nN,"sur "+L2.length+" districts",COL.gris);
 k("Cases renseignées",qp===null?"–":fmt(qp),q?(q.ren+" / "+q.cases+" cases de l'extraction"):"","#1F4E78");
 $("kpis").innerHTML=h;
}

/* ---------- carte ---------- */
var map=null,geoLayer=null,lastBounds=null,baseClair=null,baseSombre=null,baseActive=null;
var nGeo=(DATA.geojson&&DATA.geojson.features)?DATA.geojson.features.length:0;
function applyStyle(l,x){l.setStyle({fillColor:couleur(x.v),fillOpacity:x.inReg?.88:.12,weight:x.inReg?1.2:.4,color:x.inReg?(isDark()?"#dbe5f0":"#444"):"#999"});}
function fitCurrent(){try{if(map&&lastBounds&&lastBounds.isValid())map.fitBounds(lastBounds,{padding:[20,20],maxZoom:10});}catch(e){}}
function initCarte(){
 if(typeof L==="undefined"){$("map").innerHTML="<p class='note'>Carte indisponible : la bibliothèque Leaflet n'a pas pu être chargée (connexion internet requise).</p>";return;}
 map=L.map("map").setView([14.5,-14.5],7);
 baseClair=L.tileLayer("https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",{maxZoom:19,attribution:"&copy; OpenStreetMap &copy; CARTO"});
 baseSombre=L.tileLayer("https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",{maxZoom:19,attribution:"&copy; OpenStreetMap &copy; CARTO"});
 var osm=L.tileLayer("https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",{maxZoom:18,attribution:"&copy; OpenStreetMap"});
 var sat=L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",{maxZoom:18,attribution:"Tiles &copy; Esri"});
 baseActive=isDark()?baseSombre:baseClair;baseActive.addTo(map);
 L.control.layers({"Fond clair":baseClair,"Fond sombre":baseSombre,"OpenStreetMap":osm,"Satellite":sat},{}).addTo(map);
 map.on("baselayerchange",function(ev){baseActive=ev.layer;});
 var Reset=L.Control.extend({options:{position:"topleft"},onAdd:function(){var b=L.DomUtil.create("button","map-btn");b.type="button";b.title="Recentrer la carte";b.setAttribute("aria-label","Recentrer la carte");b.innerHTML="⌖";
  L.DomEvent.on(b,"click",function(ev){L.DomEvent.stopPropagation(ev);fitCurrent();});return b;}});
 map.addControl(new Reset());
 if(nGeo){
  geoLayer=L.geoJSON(DATA.geojson,{
   style:function(){return{weight:1,color:"#555",fillOpacity:.85,fillColor:COL.gris};},
   pointToLayer:function(f,latlng){return L.circleMarker(latlng,{radius:8,weight:1,color:"#333",fillOpacity:.9});},
   onEachFeature:function(f,l){l.bindTooltip("");
    l.on("click",function(){openDrawer(f.properties.d);});
    l.on("mouseover",function(){l.setStyle({weight:3});if(l.bringToFront)l.bringToFront();});
    l.on("mouseout",function(){var x=lignes[f.properties.d];if(x)applyStyle(l,x);});}
  }).addTo(map);
  try{map.fitBounds(geoLayer.getBounds());}catch(e){}
 }else{$("mapnote").textContent="Aucun contour de district disponible : la carte reste vide. Déposez un fichier districts_senegal.geojson dans le dossier Drive puis relancez.";}
 if(nGeo&&nGeo<D.length)$("mapnote").textContent=(D.length-nGeo)+" district(s) sans contour : ils figurent dans les tableaux mais pas sur la carte.";
}
function rendreCarte(){
 if(!map||!geoLayer)return;
 var byIdx={};lignes.forEach(function(x){byIdx[x.i]=x;});var b=null;
 geoLayer.eachLayer(function(l){var x=byIdx[l.feature.properties.d];if(!x)return;applyStyle(l,x);
  l.setTooltipContent("<b>"+esc(x.name)+"</b><br>"+esc(x.region)+"<br>Taux : <b>"+fmt(x.v)+"</b><br>Écart : "+fmtDelta(x.delta)+"<br><i>Cliquer pour la fiche</i>");
  if(x.inReg){var lb=l.getBounds?l.getBounds():L.latLngBounds([l.getLatLng(),l.getLatLng()]);b=b?b.extend(lb):L.latLngBounds(lb.getSouthWest(),lb.getNorthEast());}});
 lastBounds=b;fitCurrent();
}

/* ---------- priorités d'action ---------- */
function groupe(titre,items,opts){
 var h="<div class='al-g'><div class='al-t'><span>"+titre+"</span><span class='n'>"+items.length+"</span></div>";
 if(!items.length)return h+"<div class='note'>Aucun district</div></div>";
 items.slice(0,opts.max||6).forEach(function(x){
  h+="<button type='button' class='al' data-i='"+x.i+"'><span>"+esc(x.name)+" <small>"+esc(x.region)+"</small></span><span class='r' style='color:var(--"+opts.col+")'>"+(opts.delta?fmtDelta(x.delta):fmt(x.v))+"</span></button>";});
 var more=items.length-(opts.max||6),l="";
 if(more>0&&opts.voir)l+="<button type='button' data-voir='"+opts.voir+"'>+ "+more+" autre(s) → voir tout</button>";
 if(opts.copie)l+="<button type='button' data-copie='"+opts.copie+"'>⧉ Copier la liste</button>";
 return h+(l?"<div class='links'>"+l+"</div>":"")+"</div>";
}
var alerteListes={};
function rendreAlertes(c){
 var L2=lignes.filter(function(x){return x.inReg;});
 var crit=L2.filter(function(x){return x.cl==="crit";}).sort(function(a,b){return a.v-b.v;});
 var bais=L2.filter(function(x){return x.delta!==null&&x.delta<=-S.baisse;}).sort(function(a,b){return a.delta-b.delta;});
 var haus=L2.filter(function(x){return x.delta!==null&&x.delta>=S.baisse;}).sort(function(a,b){return b.delta-a.delta;});
 var exts=detExts.filter(function(x){return x<=c.e;}).slice(-3),pers=[];
 if(exts.length>=2)pers=L2.filter(function(x){return exts.every(function(ex){var v=valDistrict(ex,x.i,c.p,c.f);return v!==null&&v<S.critique;});}).sort(function(a,b){return a.v-b.v;});
 alerteListes={crit:crit,pers:pers};
 $("alHint").textContent="écart ≥ "+S.baisse+" pts · "+(c.ci!==null?"vs "+fmtDate(E[c.ci]):"sans comparaison");
 $("alertes").innerHTML=
  groupe("Districts critiques (&lt; "+S.critique+" %)",crit,{col:"crit",voir:"crit",copie:"crit"})+
  (exts.length>=2?groupe("Critiques sur les "+exts.length+" dernières extractions",pers,{col:"crit",copie:"pers"}):"")+
  groupe("Plus fortes baisses",bais,{col:"crit",delta:true})+
  groupe("Plus fortes progressions",haus,{col:"ok",delta:true});
}
function texteListe(items,titre){return titre+"\n"+items.map(function(x){return"- "+x.region+" · "+x.name+" : "+fmt(x.v);}).join("\n");}

/* ---------- onglet vue d'ensemble ---------- */
function rendreVue(c){
 rendreCarte();
 var vals=E.map(function(x,ei){return serv(ei,c.r,c.p,c.f);});
 mk("chEvo","line",{labels:E.map(fmtDate),datasets:lineSets(E,vals)},function(o){o.scales.y.min=0;o.scales.y.max=ymax(vals);o.plugins.legend.position="bottom";});
 rendreAlertes(c);rendreSynthese(c);rendreStat(c);
}

/* ---------- onglet régions et formulaires ---------- */
function barH(id,labels,vals,onPick){
 var ordre=labels.map(function(x,i){return i;}).sort(function(a,b){var va=vals[a],vb=vals[b];if(va===null)return 1;if(vb===null)return-1;return va-vb;});
 var L2=ordre.map(function(i){return cut(labels[i],40);}),V=ordre.map(function(i){return vals[i];});
 mk(id,"bar",{labels:L2,datasets:[{data:V,backgroundColor:V.map(couleur),borderRadius:4}]},function(o){
  o.indexAxis="y";o.plugins.legend.display=false;o.scales.x.min=0;o.scales.x.max=ymax(V);
  o.onClick=function(ev,els){if(els&&els.length&&onPick)onPick(ordre[els[0].index]);};});
}
function rendreReg(c){
 barH("chReg",R,R.map(function(x,ri){return serv(c.e,ri,c.p,c.f);}),function(ri){selR.value=ri;rendre();});
 barH("chForm",F,F.map(function(x,fi){return serv(c.e,c.r,c.p,fi);}),function(fi){selF.value=fi;rendre();});
 var pv=P.map(function(x,pi){return serv(c.e,c.r,pi,c.f);});
 mk("chPer","bar",{labels:P.map(function(x){return x.label;}),datasets:[{data:pv,backgroundColor:pv.map(couleur),borderRadius:4}]},function(o){o.plugins.legend.display=false;o.scales.y.min=0;o.scales.y.max=ymax(pv);});
 var h="<table class='heat'><thead><tr><th>Région</th>";
 F.forEach(function(f){h+="<th class='rot' title='"+esc(f)+"'><div>"+esc(cut(f,34))+"</div></th>";});
 h+="<th class='rot'><div>Tous formulaires</div></th></tr></thead><tbody>";
 [-1].concat(R.map(function(x,i){return i;})).forEach(function(ri){
  h+="<tr><td class='rl'>"+(ri<0?"Ensemble":esc(R[ri]))+"</td>";
  F.forEach(function(f,fi){var v=serv(c.e,ri,c.p,fi);h+="<td class='c' style='background:"+couleur(v)+";color:"+txtOn(v)+"'>"+(v===null?"–":v.toFixed(0))+"</td>";});
  var t=serv(c.e,ri,c.p,-1);h+="<td class='c' style='background:"+couleur(t)+";color:"+txtOn(t)+"'>"+(t===null?"–":t.toFixed(0))+"</td></tr>";});
 $("heat").innerHTML=h+"</tbody></table>";
 var t="<table><thead><tr><th>Région</th><th>Taux moyen</th><th>Écart</th><th>Tendance</th><th>Districts</th><th>Critiques</th><th>À surveiller</th><th>Conformes</th><th>Sans donnée</th></tr></thead><tbody>";
 [-1].concat(R.map(function(x,i){return i;})).forEach(function(ri){
  var dl=lignes.filter(function(x){return ri<0||D[x.i].r===ri;});
  var m=serv(c.e,ri,c.p,c.f),mp=c.ci!==null?serv(c.ci,ri,c.p,c.f):null,dd=(m!==null&&mp!==null)?m-mp:null;
  function n(k){return dl.filter(function(x){return x.cl===k;}).length;}
  t+="<tr class='clic' data-reg='"+ri+"'"+(ri<0?" style='font-weight:700'":"")+"><td>"+(ri<0?"Ensemble":esc(R[ri]))+"</td><td class='c' style='background:"+couleur(m)+";color:"+txtOn(m)+"'>"+fmt(m)+"</td><td>"+fmtDelta(dd)+"</td><td>"+sparkS(E.map(function(x,ei){return serv(ei,ri,c.p,c.f);}))+"</td><td>"+dl.length+"</td><td>"+n("crit")+"</td><td>"+n("warn")+"</td><td>"+n("ok")+"</td><td>"+n("nd")+"</td></tr>";});
 $("tabReg").innerHTML=t+"</tbody></table>";
 rendreRegPlus(c);
}

/* ---------- onglet évolution ---------- */
function rendreEvo(c){
 var sets=[{label:"Ensemble",data:E.map(function(x,ei){return serv(ei,-1,c.p,c.f);}),borderColor:"#111827",backgroundColor:"#111827",borderWidth:3.5,pointRadius:4,tension:.2,spanGaps:true}];
 R.forEach(function(rg,ri){var col=PAL[ri%PAL.length];sets.push({label:rg,data:E.map(function(x,ei){return serv(ei,ri,c.p,c.f);}),borderColor:col,backgroundColor:col,borderWidth:1.8,pointRadius:2.5,tension:.2,spanGaps:true});});
 var all=[];sets.forEach(function(s){all=all.concat(s.data);});
 mk("chEvoReg","line",{labels:E.map(fmtDate),datasets:sets},function(o){o.scales.y.min=0;o.scales.y.max=ymax(all);o.plugins.legend.position="bottom";});
 var h="<table><thead><tr><th>Extraction</th><th>Taux moyen</th><th>Écart vs précédente</th><th>Cases renseignées</th><th>Valeurs &gt; 100 %</th></tr></thead><tbody>";
 for(var ei=E.length-1;ei>=0;ei--){var m=serv(ei,c.r,c.p,c.f),mp=ei>0?serv(ei-1,c.r,c.p,c.f):null,dd=(m!==null&&mp!==null)?m-mp:null,q=qmap[ei]||{cases:0,ren:0,sup100:0};
  h+="<tr><td>"+fmtDate(E[ei])+"</td><td class='c' style='background:"+couleur(m)+";color:"+txtOn(m)+"'>"+fmt(m)+"</td><td>"+fmtDelta(dd)+"</td><td>"+(q.cases?fmt(100*q.ren/q.cases)+" ("+q.ren+"/"+q.cases+")":"–")+"</td><td>"+q.sup100+"</td></tr>";}
 $("tabEvo").innerHTML=h+"</tbody></table>";
}

/* ---------- onglet districts ---------- */
function spark(i,c){
 var ex=detExts.filter(function(x){return x<=c.e;}),pts=[];
 ex.forEach(function(e,k){var v=valDistrict(e,i,c.p,c.f);if(v!==null)pts.push([k,v]);});
 if(pts.length<2)return "–";
 var W=86,H=24,n=Math.max(ex.length-1,1),mx=Math.max(100,Math.max.apply(null,pts.map(function(p){return p[1];})));
 var path=pts.map(function(p){return(2+(W-4)*p[0]/n).toFixed(1)+","+(H-2-(H-4)*p[1]/mx).toFixed(1);}).join(" ");
 var last=pts[pts.length-1];
 return "<svg width='"+W+"' height='"+H+"' viewBox='0 0 "+W+" "+H+"' aria-hidden='true'><polyline points='"+path+"' fill='none' stroke='currentColor' stroke-width='1.6' opacity='.7'/><circle cx='"+(2+(W-4)*last[0]/n).toFixed(1)+"' cy='"+(H-2-(H-4)*last[1]/mx).toFixed(1)+"' r='3' fill='"+couleur(last[1])+"'/></svg>";
}
function rendreDis(c){
 var q=st.filtre.toLowerCase(),sf=selS.value;
 var rows=lignes.filter(function(x){return x.inReg&&(sf==="all"||x.cl===sf)&&(!q||(x.name+" "+x.region).toLowerCase().indexOf(q)>=0);});
 var ordreCl={crit:0,warn:1,ok:2,nd:3};
 rows.sort(function(a,b){var k=st.tri.k,va=k==="cl"?ordreCl[a.cl]:a[k],vb=k==="cl"?ordreCl[b.cl]:b[k];
  if(va===null&&vb===null)return 0;if(va===null)return 1;if(vb===null)return-1;
  if(typeof va==="string")return va.localeCompare(vb)*st.tri.dir;return(va-vb)*st.tri.dir;});
 var h="";
 rows.forEach(function(x){h+="<tr class='clic' data-i='"+x.i+"'><td>"+esc(x.region)+"</td><td>"+esc(x.name)+"</td><td class='c' style='background:"+couleur(x.v)+";color:"+txtOn(x.v)+"'>"+fmt(x.v)+"</td><td>"+fmtDelta(x.delta)+"</td><td>"+spark(x.i,c)+"</td><td><span class='pill "+x.cl+"'>"+LIB[x.cl]+"</span></td></tr>";});
 $("tb").innerHTML=h||"<tr><td colspan='6' class='note'>Aucun district</td></tr>";
 derniereVue=rows;
}

/* ---------- onglet qualité ---------- */
function rendreQua(c){
 var pc=E.map(function(x,ei){var q=qmap[ei];return q&&q.cases?100*q.ren/q.cases:null;});
 mk("chQual","line",{labels:E.map(fmtDate),datasets:[{label:"Cases renseignées (%)",data:pc,borderColor:"#1F4E78",backgroundColor:"#1F4E78",tension:.25,spanGaps:true,pointRadius:4}]},function(o){o.scales.y.min=0;o.scales.y.max=100;o.plugins.legend.display=false;});
 var h="";
 if(DG){
  h+="<div class='note'>Extraction du "+fmtDate(DG.date)+" · niveau des districts reconnu : "+esc(DG.niveau)+"</div><table><thead><tr><th>Région</th><th>Valeurs récupérées</th><th>Cases</th><th>Taux de remplissage</th></tr></thead><tbody>";
  DG.regions.forEach(function(r){var t=r.cases?100*r.valeurs/r.cases:null;h+="<tr><td>"+esc(r.region)+"</td><td>"+r.valeurs+"</td><td>"+r.cases+"</td><td class='c' style='background:"+couleur(t)+";color:"+txtOn(t)+"'>"+fmt(t)+"</td></tr>";});
  h+="</tbody></table>";
  h+="<h2 style='margin:14px 0 6px;font-size:13px'>Formulaires sans correspondance dans DHIS2 ("+DG.formulaires_ko.length+")</h2>"+(DG.formulaires_ko.length?"<div class='prose'><ul>"+DG.formulaires_ko.map(function(x){return"<li>"+esc(x)+"</li>";}).join("")+"</ul></div>":"<div class='note'>Aucun</div>");
  h+="<h2 style='margin:14px 0 6px;font-size:13px'>Districts non reconnus dans DHIS2 ("+DG.districts_ko.length+")</h2>"+(DG.districts_ko.length?"<div class='prose'><ul>"+DG.districts_ko.map(function(x){return"<li>"+esc(x)+"</li>";}).join("")+"</ul></div>":"<div class='note'>Aucun</div>");
 }else{h="<div class='note'>Aucun diagnostic disponible.</div>";}
 var q=qmap[E.length-1];if(q&&q.sup100)h+="<div class='note' style='margin-top:10px'>⚠ "+q.sup100+" valeur(s) supérieure(s) à 100 % dans la dernière extraction (rapports tardifs ou erreur de paramétrage) : à vérifier.</div>";
 $("diag").innerHTML=h;
 $("methode").innerHTML="<ul><li><b>Taux</b> : taux de rapportage DHIS2 (reporting rate) par district, période et formulaire ; moyenne simple des cases sélectionnées.</li>"+
  "<li><b>Seuils</b> : critique &lt; "+S.critique+" % ; à surveiller de "+S.critique+" à "+S.conforme+" % ; conforme ≥ "+S.conforme+" %. Une baisse ou une progression est signalée à partir de "+S.baisse+" points d'écart.</li>"+
  "<li><b>Écart</b> : différence, en points, avec l'extraction de comparaison choisie (par défaut la précédente).</li>"+
  "<li><b>Mémoire</b> : chaque extraction est conservée ; le détail par district est disponible pour les "+detExts.length+" dernière(s) extraction(s), les courbes couvrent tout l'historique.</li>"+
  "<li><b>Pas de donnée</b> : DHIS2 n'a renvoyé aucune valeur pour ce district avec les filtres choisis.</li></ul>";
}

/* ---------- fiche district ---------- */
function openDrawer(i){
 st.open=i;var c=cur,d=D[i],x=lignes[i];
 $("dTitle").textContent=d.name;$("dSub").textContent=R[d.r]+" · "+contexte(c).split(" · ").slice(2).join(" · ");
 $("dKpi").innerHTML="<div class='big' style='color:"+(x.v===null?"var(--muted)":couleur(x.v))+"'>"+fmt(x.v)+"</div><div><span class='pill "+x.cl+"'>"+LIB[x.cl]+"</span><div class='muted' style='margin-top:4px'>Écart : "+fmtDelta(x.delta)+(c.ci!==null?" vs "+fmtDate(E[c.ci]):"")+"</div></div>";
 var ex=detExts.filter(function(e){return e<=c.e;}),vals=ex.map(function(e){return valDistrict(e,i,c.p,c.f);});
 mk("chDist","line",{labels:ex.map(function(e){return fmtDate(E[e]);}),datasets:lineSets(ex,vals)},function(o){o.scales.y.min=0;o.scales.y.max=ymax(vals);o.plugins.legend.display=false;});
 var h="<table class='heat'><thead><tr><th>Formulaire</th>";
 P.forEach(function(p){h+="<th title='"+esc(p.label)+"'>"+esc(pcourt(p.label))+"</th>";});
 h+="<th>Moy.</th></tr></thead><tbody>";
 F.forEach(function(f,fi){h+="<tr><td class='rl'>"+esc(cut(f,30))+"</td>";
  P.forEach(function(p,pi){var v=valDistrict(c.e,i,pi,fi);h+="<td class='c' style='background:"+couleur(v)+";color:"+txtOn(v)+"'>"+(v===null?"–":v.toFixed(0))+"</td>";});
  var m=valDistrict(c.e,i,-1,fi);h+="<td class='c' style='background:"+couleur(m)+";color:"+txtOn(m)+"'>"+(m===null?"–":m.toFixed(0))+"</td></tr>";});
 $("dHeat").innerHTML=h+"</tbody></table>";
 $("drawer").classList.add("on");$("drawer").setAttribute("aria-hidden","false");$("scrim").classList.add("on");
 $("dClose").focus();
}
function closeDrawer(){st.open=null;$("drawer").classList.remove("on");$("drawer").setAttribute("aria-hidden","true");$("scrim").classList.remove("on");}

/* ---------- robustesse : bandeau d'information ---------- */
function ban(k,t){return "<div class='banner "+k+"'"+(k==="err"?" data-err":"")+">"+t+"</div>";}
function erreur(ou,e){console.error(e);var b=$("banner");if(b&&b.innerHTML.indexOf("data-err")<0)b.insertAdjacentHTML("beforeend",ban("err","Une section n'a pas pu s'afficher ("+esc(ou)+" : "+esc((e&&e.message)||e)+"). Le reste du tableau de bord reste utilisable."));}
function verifier(){
 var h="",last=E[E.length-1],jours=Math.floor((Date.now()-new Date(last+"T00:00:00").getTime())/864e5);
 if(jours>7)h+=ban("warn","⚠ Ces données datent de "+jours+" jours (dernière extraction : "+fmtDate(last)+"). Relancez l'extraction hebdomadaire pour les actualiser.");
 if(typeof Chart==="undefined")h+=ban("warn","Les graphiques sont indisponibles : la bibliothèque Chart.js n'a pas pu être chargée (connexion internet requise). Les tableaux restent utilisables.");
 if(E.length<2)h+=ban("info","Une seule extraction en mémoire : les écarts et les tendances apparaîtront dès la prochaine extraction.");
 var q=qmap[E.length-1];if(q&&q.cases&&q.ren/q.cases<.8)h+=ban("warn","Seulement "+fmt(100*q.ren/q.cases)+" des cases sont renseignées dans la dernière extraction : vérifier l'onglet « Qualité et méthode ».");
 $("banner").innerHTML=h;
}
var MC={"Janvier":"Janv.","Février":"Févr.","Avril":"Avr.","Juillet":"Juil.","Septembre":"Sept.","Octobre":"Oct.","Novembre":"Nov.","Décembre":"Déc."};
function pcourt(l){var a=String(l).split(" ");return(MC[a[0]]||a[0])+(a[1]?" "+a[1]:"");}
function sparkS(vals){
 var pts=[];vals.forEach(function(v,k){if(v!==null&&v!==undefined)pts.push([k,v]);});
 if(pts.length<2)return "–";
 var W=86,H=24,n=Math.max(vals.length-1,1),mx=Math.max(100,Math.max.apply(null,pts.map(function(p){return p[1];})));
 var path=pts.map(function(p){return(2+(W-4)*p[0]/n).toFixed(1)+","+(H-2-(H-4)*p[1]/mx).toFixed(1);}).join(" ");
 var last=pts[pts.length-1];
 return "<svg width='"+W+"' height='"+H+"' viewBox='0 0 "+W+" "+H+"' aria-hidden='true'><polyline points='"+path+"' fill='none' stroke='currentColor' stroke-width='1.6' opacity='.7'/><circle cx='"+(2+(W-4)*last[0]/n).toFixed(1)+"' cy='"+(H-2-(H-4)*last[1]/mx).toFixed(1)+"' r='3' fill='"+couleur(last[1])+"'/></svg>";
}

/* ---------- état partageable (lien) ---------- */
function sauverEtat(){
 try{var c=cur;if(!c)return;
  var p=["tab="+st.tab,"e="+E[c.e],"r="+(c.r<0?"":encodeURIComponent(R[c.r])),"p="+(c.p<0?"":P[c.p].code),"f="+(c.f<0?"":encodeURIComponent(F[c.f]))];
  if(selC.value!=="auto")p.push("c="+E[+selC.value]);
  history.replaceState(null,"","#"+p.join("&"));}catch(e){}
}
function chargerEtat(){
 try{var h=location.hash.replace(/^#/,"");if(!h)return;var o={};
  h.split("&").forEach(function(kv){var i=kv.indexOf("=");if(i>0)o[kv.slice(0,i)]=decodeURIComponent(kv.slice(i+1));});
  if(o.tab&&TABS.indexOf(o.tab)>=0)st.tab=o.tab;
  var ei=E.indexOf(o.e);if(ei>=0&&det[ei]){selE.value=ei;remplirCmp();}
  var ci=E.indexOf(o.c);if(ci>=0&&ci<(+selE.value)&&[].some.call(selC.options,function(x){return x.value===String(ci);}))selC.value=ci;
  if(o.r){var ri=R.indexOf(o.r);if(ri>=0)selR.value=ri;}
  if(o.p){P.forEach(function(x,i){if(x.code===o.p)selP.value=i;});}
  if(o.f){var fi=F.indexOf(o.f);if(fi>=0)selF.value=fi;}
 }catch(e){}
}

/* ---------- synthèse automatique + statuts ---------- */
function rendreSynthese(c){
 var L2=lignes.filter(function(x){return x.inReg;}),T=L2.length;
 var m=serv(c.e,c.r,c.p,c.f),mp=c.ci!==null?serv(c.ci,c.r,c.p,c.f):null,dm=(m!==null&&mp!==null)?m-mp:null;
 function n(k){return L2.filter(function(x){return x.cl===k;}).length;}
 var nc=n("crit"),nw=n("warn"),no=n("ok"),nn=n("nd"),it=[],cl=classe(m);
 it.push("<span class='pill "+cl+"'>"+LIB[cl]+"</span> Taux moyen de complétude : <b>"+fmt(m)+"</b>"+(dm===null?"":" ("+fmtDelta(dm)+" vs "+fmtDate(E[c.ci])+")")+".");
 it.push("<b>"+nc+"</b> district(s) critique(s) ("+(T?Math.round(100*nc/T):0)+" %), <b>"+nw+"</b> à surveiller, <b>"+no+"</b> conforme(s)"+(nn?", "+nn+" sans donnée":"")+" sur "+T+".");
 if(c.r<0){
  var rv=R.map(function(x,ri){return{n:x,v:serv(c.e,ri,c.p,c.f)};}).filter(function(x){return x.v!==null;}).sort(function(a,b){return a.v-b.v;});
  if(rv.length>1)it.push("Région la plus faible : <b>"+esc(rv[0].n)+"</b> ("+fmt(rv[0].v)+") ; la plus performante : <b>"+esc(rv[rv.length-1].n)+"</b> ("+fmt(rv[rv.length-1].v)+").");
 }
 if(c.f<0&&F.length>1){
  var fv=F.map(function(x,fi){return{n:x,v:serv(c.e,c.r,c.p,fi)};}).filter(function(x){return x.v!==null;}).sort(function(a,b){return a.v-b.v;});
  if(fv.length>1)it.push("Formulaire le plus faible : <b>"+esc(cut(fv[0].n,48))+"</b> ("+fmt(fv[0].v)+") ; le meilleur : <b>"+esc(cut(fv[fv.length-1].n,48))+"</b> ("+fmt(fv[fv.length-1].v)+").");
 }
 var bais=L2.filter(function(x){return x.delta!==null&&x.delta<=-S.baisse;}).sort(function(a,b){return a.delta-b.delta;});
 var haus=L2.filter(function(x){return x.delta!==null&&x.delta>=S.baisse;}).length;
 if(bais.length)it.push("Plus forte baisse : <b>"+esc(bais[0].name)+"</b> ("+esc(bais[0].region)+", "+fmtDelta(bais[0].delta)+")"+(bais.length>1?" ; "+(bais.length-1)+" autre(s) baisse(s) d'au moins "+S.baisse+" pts":"")+".");
 if(haus)it.push("<b>"+haus+"</b> district(s) en progression d'au moins "+S.baisse+" pts.");
 if(alerteListes.pers&&alerteListes.pers.length)it.push("<b>"+alerteListes.pers.length+"</b> district(s) restent critiques sur plusieurs extractions consécutives : action prioritaire recommandée.");
 $("synth").innerHTML="<ul class='synth'>"+it.map(function(x){return"<li>"+x+"</li>";}).join("")+"</ul>";
 $("synCtx").textContent=contexte(c);
}
function rendreStat(c){
 var L2=lignes.filter(function(x){return x.inReg;}),k={crit:0,warn:0,ok:0,nd:0};
 L2.forEach(function(x){k[x.cl]++;});
 mk("chStat","doughnut",{labels:[LIB.crit,LIB.warn,LIB.ok,LIB.nd],datasets:[{data:[k.crit,k.warn,k.ok,k.nd],backgroundColor:[COL.rouge,COL.orange,COL.vert,COL.gris],borderWidth:0}]},
  function(o){delete o.scales;o.cutout="62%";o.plugins.legend.position="right";});
}
function rendreRegPlus(c){
 var cnt=R.map(function(){return{crit:0,warn:0,ok:0,nd:0};});
 lignes.forEach(function(x){cnt[D[x.i].r][x.cl]++;});
 var ordre=R.map(function(x,i){return i;}).sort(function(a,b){return cnt[b].crit-cnt[a].crit;});
 function ds(k,l,col){return{label:l,data:ordre.map(function(i){return cnt[i][k];}),backgroundColor:col,borderWidth:0};}
 mk("chStatReg","bar",{labels:ordre.map(function(i){return cut(R[i],28);}),datasets:[ds("crit",LIB.crit,COL.rouge),ds("warn",LIB.warn,COL.orange),ds("ok",LIB.ok,COL.vert),ds("nd",LIB.nd,COL.gris)]},
  function(o){o.indexAxis="y";o.scales.x.stacked=true;o.scales.y.stacked=true;o.plugins.legend.position="bottom";});
 var rows=F.map(function(f,fi){
  var m=serv(c.e,c.r,c.p,fi),mp=c.ci!==null?serv(c.ci,c.r,c.p,fi):null,nc=0;
  lignes.forEach(function(x){if(!x.inReg)return;var v=valDistrict(c.e,x.i,c.p,fi);if(v!==null&&v<S.critique)nc++;});
  return{fi:fi,name:f,m:m,d:(m!==null&&mp!==null)?m-mp:null,nc:nc};});
 rows.sort(function(a,b){if(a.m===null)return 1;if(b.m===null)return-1;return a.m-b.m;});
 var h="<table><thead><tr><th>Formulaire</th><th>Taux</th><th>Écart</th><th>Districts critiques</th></tr></thead><tbody>";
 rows.forEach(function(x){h+="<tr class='clic' data-f='"+x.fi+"'><td>"+esc(x.name)+"</td><td class='c' style='background:"+couleur(x.m)+";color:"+txtOn(x.m)+"'>"+fmt(x.m)+"</td><td>"+fmtDelta(x.d)+"</td><td>"+x.nc+"</td></tr>";});
 $("tabForm").innerHTML=h+"</tbody></table>";
 var vv=lignes.filter(function(x){return x.inReg&&x.v!==null;});
 function liste(a){return a.length?a.map(function(x){return "<button type='button' class='al' data-i='"+x.i+"'><span>"+esc(x.name)+" <small>"+esc(x.region)+"</small></span><span class='r' style='color:var(--"+x.cl+")'>"+fmt(x.v)+"</span></button>";}).join(""):"<div class='note'>Aucun district</div>";}
 $("top10").innerHTML=liste(vv.slice().sort(function(a,b){return b.v-a.v;}).slice(0,10));
 $("flop10").innerHTML=liste(vv.slice().sort(function(a,b){return a.v-b.v;}).slice(0,10));
}

/* ---------- rendu global ---------- */
function rendre(){
 var c;
 try{c=calculer();rendreKpis(c);}catch(e){erreur("calcul",e);return;}
 try{
  if(st.tab==="vue")rendreVue(c);else if(st.tab==="reg")rendreReg(c);else if(st.tab==="evo")rendreEvo(c);else if(st.tab==="dis")rendreDis(c);else rendreQua(c);
 }catch(e){erreur("onglet "+st.tab,e);}
 try{if(st.open!==null&&$("drawer").classList.contains("on"))openDrawer(st.open);}catch(e){erreur("fiche district",e);}
 sauverEtat();
}
function show(t){
 st.tab=t;
 TABS.forEach(function(k){var on=k===t;$("p-"+k).classList.toggle("on",on);var b=document.querySelector("[data-tab='"+k+"']");b.classList.toggle("on",on);b.setAttribute("aria-selected",on?"true":"false");});
 rendre();
 if(t==="vue"&&map)setTimeout(function(){map.invalidateSize();fitCurrent();},60);
}
function setTheme(t){
 document.documentElement.setAttribute("data-theme",t);
 $("btnTheme").textContent=t==="dark"?"☀ Mode clair":"☾ Mode sombre";
 try{localStorage.setItem("tdb-theme",t);}catch(e){}
 if(map&&baseActive&&(baseActive===baseClair||baseActive===baseSombre)){map.removeLayer(baseActive);baseActive=t==="dark"?baseSombre:baseClair;baseActive.addTo(map);}
 rendre();
}

/* ---------- événements ---------- */
[selE,selC,selR,selP,selF].forEach(function(s){s.addEventListener("change",function(){if(s===selE)remplirCmp();rendre();});});
selS.addEventListener("change",rendre);
$("q").addEventListener("input",function(ev){st.filtre=ev.target.value;rendre();});
Array.prototype.forEach.call(document.querySelectorAll(".tab"),function(b){b.addEventListener("click",function(){show(b.getAttribute("data-tab"));});});
Array.prototype.forEach.call(document.querySelectorAll("th[data-k]"),function(th2){th2.addEventListener("click",function(){var k=th2.getAttribute("data-k");st.tri.dir=(st.tri.k===k)?-st.tri.dir:1;st.tri.k=k;rendre();});});
$("btnReset").addEventListener("click",function(){selE.value=detExts[detExts.length-1];remplirCmp();selC.value="auto";selR.value=-1;selP.value=-1;selF.value=-1;selS.value="all";$("q").value="";st.filtre="";rendre();});
$("btnTheme").addEventListener("click",function(){setTheme(isDark()?"light":"dark");});
$("btnPrint").addEventListener("click",function(){window.print();});
$("dClose").addEventListener("click",closeDrawer);$("scrim").addEventListener("click",closeDrawer);
document.addEventListener("keydown",function(ev){if(ev.key==="Escape")closeDrawer();});
$("alertes").addEventListener("click",function(ev){
 var b=ev.target.closest("button");if(!b)return;
 if(b.hasAttribute("data-i")){openDrawer(+b.getAttribute("data-i"));}
 else if(b.hasAttribute("data-voir")){selS.value=b.getAttribute("data-voir");st.tri={k:"v",dir:1};show("dis");}
 else if(b.hasAttribute("data-copie")){var k=b.getAttribute("data-copie"),c=cur;copier(texteListe(alerteListes[k],(k==="pers"?"Districts critiques sur plusieurs extractions – ":"Districts critiques – ")+contexte(c)));}
});
$("tb").addEventListener("click",function(ev){var tr=ev.target.closest("tr");if(tr&&tr.hasAttribute("data-i"))openDrawer(+tr.getAttribute("data-i"));});
$("tabReg").addEventListener("click",function(ev){var tr=ev.target.closest("tr");if(tr&&tr.hasAttribute("data-reg")){selR.value=tr.getAttribute("data-reg");rendre();}});
$("csv").addEventListener("click",function(){
 var out=["Région;District;Taux (%);Écart (pts);Statut"];
 derniereVue.forEach(function(x){out.push([x.region,x.name,csvNum(x.v),csvNum(x.delta),LIB[x.cl]].join(";"));});
 telecharger("districts_completude.csv",out);
});

/* ---------- démarrage ---------- */
var themeInit="light";
try{var sv=localStorage.getItem("tdb-theme");if(sv==="dark"||sv==="light")themeInit=sv;else if(window.matchMedia&&window.matchMedia("(prefers-color-scheme: dark)").matches)themeInit="dark";}catch(e){}
document.documentElement.setAttribute("data-theme",themeInit);
$("btnTheme").textContent=themeInit==="dark"?"☀ Mode clair":"☾ Mode sombre";
try{initCarte();}catch(e){erreur("carte",e);}
$("legMap").innerHTML=$("legHeat").innerHTML="<span><i style='background:"+COL.rouge+"'></i>&lt; "+S.critique+" % critique</span><span><i style='background:"+COL.orange+"'></i>"+S.critique+"–"+S.conforme+" % à surveiller</span><span><i style='background:"+COL.vert+"'></i>≥ "+S.conforme+" % conforme</span><span><i style='background:"+COL.gris+"'></i>pas de donnée</span>";
$("btnLink").addEventListener("click",function(){sauverEtat();copier(location.href,"Lien copié");});
["top10","flop10"].forEach(function(id){$(id).addEventListener("click",function(ev){var b=ev.target.closest("button");if(b&&b.hasAttribute("data-i"))openDrawer(+b.getAttribute("data-i"));});});
$("tabForm").addEventListener("click",function(ev){var tr=ev.target.closest("tr");if(tr&&tr.hasAttribute("data-f")){selF.value=tr.getAttribute("data-f");rendre();}});
chargerEtat();verifier();show(st.tab);
})();
</script></body></html>'''

def construire_dashboard(hist, dist, features, chemins, titre, nb_detail, options):
    data = preparer_donnees(hist, dist, features, nb_detail, options)
    js = json.dumps(data, ensure_ascii=False, allow_nan=False, separators=(",", ":")).replace("</", "<\\/")
    page = (HTML_MODELE.replace("__TITRE__", htmllib.escape(titre))
                       .replace("__ENTETE__", entete_html(options["entete"], options.get("logo")))
                       .replace("__GENERE__", datetime.now().strftime("%d/%m/%Y à %H:%M"))
                       .replace("__DATA__", js))
    for c in chemins:
        os.makedirs(os.path.dirname(c), exist_ok=True)
        with open(c, "w", encoding="utf-8") as fh: fh.write(page)
    return page

dist_df, features = charger_contours(s, BASE_URL, niveau, hist, GEO_CACHE, GEO_LOCAL, RECHARGER_CONTOURS)
OPTIONS_DASH = {"entete": ENTETE, "logo": LOGO, "seuils": (SEUIL_CRITIQUE, SEUIL_CONFORME, SEUIL_BAISSE),
                "diag": {"date": date_ext, "niveau": niveau, "regions": DIAG_REGIONS,
                         "formulaires_ko": sorted(col_ko), "districts_ko": sorted(dist_ko)}}
sorties = [DASHBOARD]
if ARCHIVER_DASHBOARD:
    sorties.append(f"{DOSSIER}/Archives_dashboard/Dashboard_completude_{aujourdhui.strftime('%d_%m_%Y')}.html")
construire_dashboard(hist, dist_df, features, sorties, TITRE_DASHBOARD, NB_EXTRACTIONS_DETAIL, OPTIONS_DASH)
print(f"📊 Dashboard unique mis à jour : {DASHBOARD}")
print("   (ouvrez le fichier .html dans votre navigateur ; internet requis pour le fond de carte)")

files.download(SORTIE)
files.download(DASHBOARD)

Mounted at /content/drive
